# Netflix Analysis:

Data Science Question: What region has had the highest catalogue growth of of movies and shows on Netflix from 2016 - 2021?

Definitions:
- Regions: United States and Canada (UCAN), Europe, Middle East and Africa (EMEA), Latin America (LATAM), and Asia-Pacific (APAC)
- Highest Catalogue Growth: % growth of catalogue between 2016 and 2021

In [29]:
# %pip install countryinfo[fuzzy]

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import countryinfo

In [31]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [32]:
df = pd.read_csv("netflix_titles.csv")
df.head(30)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...


In [33]:
df.groupby('type')['title'].count().sort_values(ascending=False).head(10)

type
Movie      6131
TV Show    2676
Name: title, dtype: int64

In [34]:
x = df['date_added'].apply(
    lambda x: x[-4:] if isinstance(x, str) and len(x) >= 4 else None
)

x.value_counts()

date_added
2019    2016
2020    1879
2018    1649
2021    1498
2017    1188
2016     429
2015      82
2014      24
2011      13
2013      11
2012       3
2009       2
2008       2
2010       1
Name: count, dtype: int64

## Get 2016 and 2021 Datasets
The 2021 dataset has all titles in the dataset, since 2021 is the most recent year in the dataset. The 2016 dataset would have all titles added in 2016 or before.

In [35]:
df_21 = df

df_16 = df[df['date_added'].apply(
    lambda x: int(x[-4:]) <= 2016 if isinstance(x, str) and len(x) >= 4 else False
)]

In [36]:
df_16['date_added'].apply(
    lambda x: int(x[-4:]) 
).value_counts()

date_added
2016    429
2015     82
2014     24
2011     13
2013     11
2012      3
2009      2
2008      2
2010      1
Name: count, dtype: int64

## Get 4 Regions and Unavailable Region into DataFrames

Create dataframes for each region

In [37]:
latam_df = pd.read_csv("LATAM.csv")
emea_df = pd.read_csv("EMEA.csv")
apac_df = pd.read_csv("APAC.csv")

In [38]:
latam_df.head(10)

,Country
0,Brazil
1,Mexico
2,Colombia
3,Argentina
4,Peru
5,Venezuela
6,Chile
7,Guatemala
8,Ecuador
9,Bolivia


In [39]:
emea_df.head(10)

,Country
0,Albania
1,Algeria
2,Andorra
3,Angola
4,Austria
5,Bahrain
6,Belarus
7,Belgium
8,Benin
9,Bosnia and Herzegovina


In [40]:
apac_df.head(30)

,Country
0,Australia
1,Bangladesh
2,Brunei
3,Burma (Myanmar)*
4,Cambodia
5,China
6,Cook Islands
7,Fiji
8,French Polynesia**
9,India


## Get Count of Titles by Country from Dataset
Note: Some titles have several countries listed in the "country" section of their record. These titles will be added to the count of each country listed for their record in the dataset.

In [41]:
country_counts_series = df.groupby("country")['title'].count().sort_values(ascending=False)
country_counts_series.head(10)

country
United States     2818
India              972
United Kingdom     419
Japan              245
South Korea        199
Canada             181
Spain              145
France             124
Mexico             110
Egypt              106
Name: title, dtype: int64

In [42]:
country_list = list(country_counts_series.index)

country_list.sort()

In [43]:
def get_counts_by_country(df):
    '''Get counts of titles by country, splitting entries with multiple countries into individual counts.
    df: DataFrame containing Netflix titles with a 'country' column.

    returns: Series with counts of titles by country.
    '''
    country_counts_series = df.groupby("country")['title'].count().sort_values(ascending=False)
    for entry in country_counts_series.index:
        if ", " in entry:
            count = country_counts_series[entry]
            countries_in_entry = entry.split(", ")
            #need to check and ensure each country in countries_in_entry is in the index before adding to the count, otherwise it will throw a KeyError
            for i in countries_in_entry:
                if i not in country_counts_series.index:
                    country_counts_series.loc[i] = 0
            country_counts_series.loc[countries_in_entry] += count
            
            country_counts_series.drop(entry, inplace=True)
    return country_counts_series.sort_values(ascending=False)

In [44]:
country_counts_21 = get_counts_by_country(df_21)
country_counts_16 = get_counts_by_country(df_16)

In [45]:
country_counts_16

country
United States           311
United Kingdom           66
Canada                   40
France                   40
Japan                    30
Taiwan                   24
Spain                    18
Germany                  16
India                    13
Australia                13
Mexico                   12
South Korea              11
China                    10
Netherlands               9
Norway                    7
Ireland                   7
Denmark                   7
Belgium                   7
New Zealand               6
Italy                     5
Greece                    5
Hong Kong                 4
Pakistan                  4
Turkey                    4
Argentina                 3
Israel                    3
Peru                      3
South Africa              2
Hungary                   2
Sweden                    2
Brazil                    2
Colombia                  2
Nigeria                   2
Chile                     2
Switzerland               1
Croatia     

## Counts for Each Region

### 2021 Counts
The 2021 count is the total of all titles on netflix by region, since 2021 is the most recent year in the dataset

In [46]:
ucan_count_21 = 0
latam_count_21 = 0
emea_count_21 = 0
apac_count_21 = 0

In [47]:
ucan_count_21 = country_counts_21[['United States', 'Canada']].sum()

latam_count_21 = country_counts_21[country_counts_21.index.isin(latam_df['Country'])].sum()
emea_count_21 = country_counts_21[country_counts_21.index.isin(emea_df['Country'])].sum()
apac_count_21 = country_counts_21[country_counts_21.index.isin(apac_df['Country'])].sum()


In [48]:
print(f"UCAN: {ucan_count_21}, LATAM: {latam_count_21}, EMEA: {emea_count_21}, APAC: {apac_count_21}")

UCAN: 4134, LATAM: 475, EMEA: 2854, APAC: 2305


### 2016 Counts
The 2016 count is the total of all titles on Netflix added in 2016 or before

In [52]:
ucan_count_16 = 0
latam_count_16 = 0
emea_count_16 = 0
apac_count_16 = 0

In [53]:
ucan_count_16 = country_counts_16[['United States', 'Canada']].sum()

latam_count_16 = country_counts_16[country_counts_16.index.isin(latam_df['Country'])].sum()
emea_count_16 = country_counts_16[country_counts_16.index.isin(emea_df['Country'])].sum()
apac_count_16 = country_counts_16[country_counts_16.index.isin(apac_df['Country'])].sum()


In [54]:
print(f"UCAN: {ucan_count_16}, LATAM: {latam_count_16}, EMEA: {emea_count_16}, APAC: {apac_count_16}")

UCAN: 351, LATAM: 26, EMEA: 217, APAC: 88


## Ratios for Each Region

In [61]:
ucan_ratio = round(ucan_count_21 / ucan_count_16 * 100)
latam_ratio = round(latam_count_21 / latam_count_16 * 100)
apac_ratio = round(apac_count_21 / apac_count_16 * 100)
emea_ratio = round(emea_count_21 / emea_count_16 * 100)

In [64]:
print((f"UCAN: {ucan_ratio}%\nLATAM: {latam_ratio}%\nEMEA: {emea_ratio}%\nAPAC: {apac_ratio}%"))

UCAN: 1178%
LATAM: 1827%
EMEA: 1315%
APAC: 2619%


## Final Conclusion:

The Asia-Pacific region has had the highest growth in their catalogue of Netflix titles from 2016 to 2021